In [1]:
import jax
import jax.numpy as jnp
from jax import random, grad, jit, vmap
from flax import linen as nn
import optax
import numpy as np

# ==========================================
# 1. Configuration and Architecture
# ==========================================

# Data Path (Hardcoded based on instructions)
DATA_PATH = '/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Results/NM_codes/Burgers_Periodic_SinPiX/2025_11_22_1730/burgers.npz'

# Hyperparameters
SEED = 1234
LEARNING_RATE = 1e-3
ITERATIONS = 20000
N_COLLOCATION = 10000  # Number of collocation points (Physics)
N_IC = 1000            # Number of Initial Condition points
N_BC = 1000            # Number of Boundary Condition points (Periodic)
LAYERS = [2] + [32] * 4 + [1] # Input (t,x) -> Hidden -> Output u

# MLP Architecture using Flax
class PINN(nn.Module):
    features: list

    @nn.compact
    def __call__(self, t, x):
        # Concatenate inputs
        inputs = jnp.concatenate([t, x], axis=1)
        
        # Hidden layers
        y = inputs
        for feat in self.features[:-1]:
            y = nn.Dense(feat)(y)
            y = nn.tanh(y)
            
        # Output layer
        y = nn.Dense(self.features[-1])(y)
        return y

# ==========================================
# 2. Physics and Loss Functions
# ==========================================

# Helper to apply model to scalar inputs for grad calculation
def predict_u(params, model, t, x):
    # Expand dims to match batch expectation of model (1, 1)
    return model.apply(params, t[None, :], x[None, :])[0, 0]

# Vectorized prediction for batch evaluation
def batch_predict(params, model, t, x):
    return model.apply(params, t, x)

# Residual calculation (u_t + u * u_x = 0)
def residual_fn(params, model, t, x):
    u = predict_u(params, model, t, x)
    
    # Gradients
    u_t = grad(predict_u, argnums=2)(params, model, t, x)
    u_x = grad(predict_u, argnums=3)(params, model, t, x)
    
    return u_t + u * u_x

# Vectorize the residual function
residual_vmap = vmap(residual_fn, in_axes=(None, None, 0, 0))

# Loss Function
def loss_fn(params, model, batch_col, batch_ic, batch_bc):
    # 1. Physics Loss (Residuals)
    t_c, x_c = batch_col[:, 0:1], batch_col[:, 1:2]
    res = residual_vmap(params, model, t_c, x_c)
    loss_physics = jnp.mean(res ** 2)

    # 2. Initial Condition Loss
    # u(x, 0) = -sin(pi * x)
    t_ic, x_ic = batch_ic[:, 0:1], batch_ic[:, 1:2]
    u_pred_ic = batch_predict(params, model, t_ic, x_ic)
    u_exact_ic = -jnp.sin(jnp.pi * x_ic)
    loss_ic = jnp.mean((u_pred_ic - u_exact_ic) ** 2)

    # 3. Periodic Boundary Condition Loss
    # u(-1, t) = u(1, t)
    t_bc = batch_bc[:, 0:1] # Sampled times
    x_left = -1.0 * jnp.ones_like(t_bc)
    x_right = 1.0 * jnp.ones_like(t_bc)
    
    u_left = batch_predict(params, model, t_bc, x_left)
    u_right = batch_predict(params, model, t_bc, x_right)
    loss_bc = jnp.mean((u_left - u_right) ** 2)

    return loss_physics + loss_ic + loss_bc

@jit
def update_step(params, opt_state, batch_col, batch_ic, batch_bc, optimizer, model):
    grads = grad(loss_fn)(params, model, batch_col, batch_ic, batch_bc)
    updates, new_opt_state = optimizer.update(grads, opt_state)
    new_params = optax.apply_updates(params, updates)
    return new_params, new_opt_state

# ==========================================
# 3. Data Loading and Generation
# ==========================================

def load_reference_data(path):
    print(f"Loading data from {path}...")
    try:
        data = np.load(path)
        t_coords, x_coords, u_sol = data['t'], data['x'], data['usol']
        
        # Create a meshgrid and flatten it for validation during training
        T_test, X_test = np.meshgrid(t_coords, x_coords, indexing='ij')
        t_test = T_test.flatten()[:, None]
        x_test = X_test.flatten()[:, None]
        u_test = u_sol.flatten()[:, None]
        
        # Store in a dictionary for easy access
        test_data = {
            't': jnp.array(t_test),
            'x': jnp.array(x_test),
            'u': jnp.array(u_test)
        }
        return test_data
    except FileNotFoundError:
        print("Data file not found. Please check path.")
        raise

def generate_training_data(key, n_col, n_ic, n_bc):
    key1, key2, key3 = random.split(key, 3)

    # Collocation points: t in [0, 1], x in [-1, 1]
    t_c = random.uniform(key1, (n_col, 1), minval=0.0, maxval=1.0)
    x_c = random.uniform(key1, (n_col, 1), minval=-1.0, maxval=1.0) # Reusing key1 intentionally for joint sampling if not using LHS
    # Ideally use separate keys or combine:
    t_c = random.uniform(key1, (n_col, 1), minval=0.0, maxval=1.0)
    x_c = random.uniform(key2, (n_col, 1), minval=-1.0, maxval=1.0)
    batch_col = jnp.concatenate([t_c, x_c], axis=1)

    # IC points: t = 0, x in [-1, 1]
    t_ic = jnp.zeros((n_ic, 1))
    x_ic = random.uniform(key3, (n_ic, 1), minval=-1.0, maxval=1.0)
    batch_ic = jnp.concatenate([t_ic, x_ic], axis=1)

    # BC points (Periodic): t in [0, 1]
    # We only need time coordinates, x is fixed to -1 and 1 in loss function
    key4 = random.split(key3)[0]
    t_bc = random.uniform(key4, (n_bc, 1), minval=0.0, maxval=1.0)
    # x placeholder not strictly needed here but kept for structure if desired
    x_bc_placeholder = jnp.zeros((n_bc, 1)) 
    batch_bc = jnp.concatenate([t_bc, x_bc_placeholder], axis=1)

    return batch_col, batch_ic, batch_bc

# ==========================================
# 4. Main Execution
# ==========================================

def main():
    # Load Validation Data
    test_data = load_reference_data(DATA_PATH)
    
    # Initialization
    key = random.PRNGKey(SEED)
    model = PINN(features=LAYERS)
    
    # Initialize parameters
    key, subkey = random.split(key)
    dummy_t = jnp.ones((1, 1))
    dummy_x = jnp.ones((1, 1))
    params = model.init(subkey, dummy_t, dummy_x)
    
    # Optimizer (JAX native only)
    optimizer = optax.adam(LEARNING_RATE)
    opt_state = optimizer.init(params)
    
    # Generate Training Data
    key, subkey = random.split(key)
    batch_col, batch_ic, batch_bc = generate_training_data(subkey, N_COLLOCATION, N_IC, N_BC)
    
    print("Starting training...")
    print("-" * 60)
    print(f"{'Iter':<10} | {'Loss':<15} | {'L2 Relative Error':<15}")
    print("-" * 60)

    # Training Loop
    for it in range(ITERATIONS + 1):
        params, opt_state = update_step(params, opt_state, batch_col, batch_ic, batch_bc, optimizer, model)
        
        # Logging
        if it % 500 == 0:
            # Compute current loss
            current_loss = loss_fn(params, model, batch_col, batch_ic, batch_bc)
            
            # Compute L2 Relative Error on Test Data
            u_pred_test = batch_predict(params, model, test_data['t'], test_data['x'])
            error_u = jnp.linalg.norm(test_data['u'] - u_pred_test, 2) / jnp.linalg.norm(test_data['u'], 2)
            
            print(f"{it:<10} | {current_loss:.4e}      | {error_u:.4e}")

    print("-" * 60)
    print("Training finished.")

if __name__ == '__main__':
    main()

Loading data from /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Results/NM_codes/Burgers_Periodic_SinPiX/2025_11_22_1730/burgers.npz...
Starting training...
------------------------------------------------------------
Iter       | Loss            | L2 Relative Error
------------------------------------------------------------


TypeError: Error interpreting argument to <function update_step at 0x7f18c4de1ee0> as an abstract array. The problematic value is of type <class 'function'> and was passed to the function at path optimizer.init.
This typically means that a jit-wrapped function was called with a non-array argument, and this argument was not marked as static using the static_argnums or static_argnames parameters of jax.jit.

In [2]:
import jax
import jax.numpy as jnp
from jax import random, grad, jit, vmap
from flax import linen as nn
import optax
import numpy as np
from functools import partial

# ==========================================
# 1. Configuration and Architecture
# ==========================================

# Data Path
DATA_PATH = '/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Results/NM_codes/Burgers_Periodic_SinPiX/2025_11_22_1730/burgers.npz'

# Hyperparameters
SEED = 1234
LEARNING_RATE = 1e-3
ITERATIONS = 20000
N_COLLOCATION = 10000
N_IC = 1000
N_BC = 1000
LAYERS = [2] + [32] * 4 + [1]

class PINN(nn.Module):
    features: list

    @nn.compact
    def __call__(self, t, x):
        inputs = jnp.concatenate([t, x], axis=1)
        y = inputs
        for feat in self.features[:-1]:
            y = nn.Dense(feat)(y)
            y = nn.tanh(y)
        y = nn.Dense(self.features[-1])(y)
        return y

# ==========================================
# 2. Physics and Loss Functions
# ==========================================

def predict_u(params, model, t, x):
    # Expand dims to match batch expectation
    return model.apply(params, t[None, :], x[None, :])[0, 0]

def batch_predict(params, model, t, x):
    return model.apply(params, t, x)

def residual_fn(params, model, t, x):
    u = predict_u(params, model, t, x)
    u_t = grad(predict_u, argnums=2)(params, model, t, x)
    u_x = grad(predict_u, argnums=3)(params, model, t, x)
    return u_t + u * u_x

residual_vmap = vmap(residual_fn, in_axes=(None, None, 0, 0))

def loss_fn(params, model, batch_col, batch_ic, batch_bc):
    # 1. Physics Loss
    t_c, x_c = batch_col[:, 0:1], batch_col[:, 1:2]
    res = residual_vmap(params, model, t_c, x_c)
    loss_physics = jnp.mean(res ** 2)

    # 2. Initial Condition Loss
    t_ic, x_ic = batch_ic[:, 0:1], batch_ic[:, 1:2]
    u_pred_ic = batch_predict(params, model, t_ic, x_ic)
    u_exact_ic = -jnp.sin(jnp.pi * x_ic)
    loss_ic = jnp.mean((u_pred_ic - u_exact_ic) ** 2)

    # 3. Periodic BC Loss
    t_bc = batch_bc[:, 0:1]
    x_left = -1.0 * jnp.ones_like(t_bc)
    x_right = 1.0 * jnp.ones_like(t_bc)
    u_left = batch_predict(params, model, t_bc, x_left)
    u_right = batch_predict(params, model, t_bc, x_right)
    loss_bc = jnp.mean((u_left - u_right) ** 2)

    return loss_physics + loss_ic + loss_bc

# FIXED: Added static_argnums to tell JAX to ignore optimizer (5) and model (6) during tracing
@partial(jit, static_argnums=(5, 6))
def update_step(params, opt_state, batch_col, batch_ic, batch_bc, optimizer, model):
    grads = grad(loss_fn)(params, model, batch_col, batch_ic, batch_bc)
    updates, new_opt_state = optimizer.update(grads, opt_state)
    new_params = optax.apply_updates(params, updates)
    return new_params, new_opt_state

# ==========================================
# 3. Data Loading and Main
# ==========================================

def load_reference_data(path):
    print(f"Loading data from {path}...")
    try:
        data = np.load(path)
        t_coords, x_coords, u_sol = data['t'], data['x'], data['usol']
        T_test, X_test = np.meshgrid(t_coords, x_coords, indexing='ij')
        return {
            't': jnp.array(T_test.flatten()[:, None]),
            'x': jnp.array(X_test.flatten()[:, None]),
            'u': jnp.array(u_sol.flatten()[:, None])
        }
    except FileNotFoundError:
        print("Data file not found.")
        raise

def generate_training_data(key, n_col, n_ic, n_bc):
    key1, key2, key3, key4 = random.split(key, 4)
    
    # Collocation
    t_c = random.uniform(key1, (n_col, 1), minval=0.0, maxval=1.0)
    x_c = random.uniform(key2, (n_col, 1), minval=-1.0, maxval=1.0)
    batch_col = jnp.concatenate([t_c, x_c], axis=1)

    # IC
    t_ic = jnp.zeros((n_ic, 1))
    x_ic = random.uniform(key3, (n_ic, 1), minval=-1.0, maxval=1.0)
    batch_ic = jnp.concatenate([t_ic, x_ic], axis=1)

    # BC
    t_bc = random.uniform(key4, (n_bc, 1), minval=0.0, maxval=1.0)
    x_bc_placeholder = jnp.zeros((n_bc, 1)) 
    batch_bc = jnp.concatenate([t_bc, x_bc_placeholder], axis=1)

    return batch_col, batch_ic, batch_bc

def main():
    test_data = load_reference_data(DATA_PATH)
    
    # Setup
    key = random.PRNGKey(SEED)
    model = PINN(features=LAYERS)
    
    # Init params
    key, subkey = random.split(key)
    dummy_t = jnp.ones((1, 1))
    dummy_x = jnp.ones((1, 1))
    params = model.init(subkey, dummy_t, dummy_x)
    
    # Optimizer
    optimizer = optax.adam(LEARNING_RATE)
    opt_state = optimizer.init(params)
    
    # Data
    key, subkey = random.split(key)
    batch_col, batch_ic, batch_bc = generate_training_data(subkey, N_COLLOCATION, N_IC, N_BC)
    
    print("Starting training...")
    print("-" * 65)
    print(f"{'Iter':<10} | {'Loss':<15} | {'L2 Rel Error':<15}")
    print("-" * 65)

    for it in range(ITERATIONS + 1):
        # Pass optimizer and model as arguments (handled as static by jit now)
        params, opt_state = update_step(
            params, opt_state, batch_col, batch_ic, batch_bc, optimizer, model
        )
        
        if it % 500 == 0:
            current_loss = loss_fn(params, model, batch_col, batch_ic, batch_bc)
            u_pred_test = batch_predict(params, model, test_data['t'], test_data['x'])
            error_u = jnp.linalg.norm(test_data['u'] - u_pred_test, 2) / jnp.linalg.norm(test_data['u'], 2)
            print(f"{it:<10} | {current_loss:.4e}      | {error_u:.4e}")

if __name__ == '__main__':
    main()

Loading data from /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Results/NM_codes/Burgers_Periodic_SinPiX/2025_11_22_1730/burgers.npz...
Starting training...
-----------------------------------------------------------------
Iter       | Loss            | L2 Rel Error   
-----------------------------------------------------------------


ValueError: Non-hashable static arguments are not supported. An error occurred while trying to hash an object of type <class '__main__.PINN'>, PINN(
    # attributes
    features = [2, 32, 32, 32, 32, 1]
). The error was:
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/users/jdtoscan/.conda/envs/JAX_311/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
  File "/users/jdtoscan/.conda/envs/JAX_311/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
  File "/users/jdtoscan/.conda/envs/JAX_311/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
  File "/users/jdtoscan/.conda/envs/JAX_311/lib/python3.11/site-packages/tornado/platform/asyncio.py", line 205, in start
  File "/users/jdtoscan/.conda/envs/JAX_311/lib/python3.11/asyncio/base_events.py", line 608, in run_forever
  File "/users/jdtoscan/.conda/envs